# HMC Production Notebook

Set `SUMMARY_JSON` below to one of the summaries produced by `test/production_hmc.py`:

- `production_stage.json`
- `production_tune.json`
- `production_benchmark.json`

The notebook will switch plotting logic automatically based on `summary["mode"]`.


In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

SUMMARY_JSON = Path("production_stage.json")
CASE_SELECT = None
TRACE_OBSERVABLES = ["squareOcc", "IPR", "doubleOcc", "nearestOcc"]

summary = json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))
mode = summary.get("mode", "")
work_root = SUMMARY_JSON.resolve().parent
report_dir = work_root / "report"
print(f"mode={mode}")
print(f"work_root={work_root}")


In [ ]:
def display_report_markdown(path: Path) -> None:
    if not path.exists():
        print(f"report not found: {path}")
        return
    text = path.read_text(encoding="utf-8")
    parent = path.parent.as_posix()
    text = re.sub(r"!\[([^\]]*)\]\(([^)]+)\)", lambda m: f"![{m.group(1)}]({parent}/{m.group(2)})", text)
    display(Markdown(text))

display_report_markdown(report_dir / "report.md")


In [ ]:
if mode == "stage":
    stage_cases = pd.read_csv(work_root / "production_stage_cases.csv")
    stage_observables = pd.read_csv(work_root / "production_stage_observables.csv")
    stage_samples = pd.read_csv(work_root / "production_stage_samples.csv")
    display(stage_cases)
elif mode == "tune":
    tune_cases = summary["cases"]
    display(pd.DataFrame([{**{"name": case["name"]}, **case["recommended"]} for case in tune_cases]))
elif mode == "benchmark":
    bench_cases = pd.read_csv(work_root / "production_benchmark_cases.csv")
    bench_obs = pd.read_csv(work_root / "production_benchmark_observables.csv")
    display(bench_cases)
else:
    raise ValueError(f"unsupported mode: {mode}")


In [ ]:
if mode == "stage":
    labels = stage_cases["name"].tolist()
    x = np.arange(len(labels))
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
    axes[0, 0].bar(x, stage_cases["acceptance_mean"], color="#3b6fb6")
    axes[0, 0].errorbar(x, stage_cases["acceptance_mean"], yerr=stage_cases["acceptance_stderr"], fmt="none", ecolor="black", capsize=4)
    axes[0, 0].set_title("Acceptance")
    axes[0, 1].bar(x, stage_cases["tau_int_doubleOcc_mean"], color="#009e73")
    axes[0, 1].errorbar(x, stage_cases["tau_int_doubleOcc_mean"], yerr=stage_cases["tau_int_doubleOcc_stderr"], fmt="none", ecolor="black", capsize=4)
    axes[0, 1].set_title("tau_int(doubleOcc)")
    axes[1, 0].bar(x, stage_cases["ess_per_sec_doubleOcc_mean"], color="#d55e00")
    axes[1, 0].errorbar(x, stage_cases["ess_per_sec_doubleOcc_mean"], yerr=stage_cases["ess_per_sec_doubleOcc_stderr"], fmt="none", ecolor="black", capsize=4)
    axes[1, 0].set_title("ESS/sec")
    width = 0.38
    axes[1, 1].bar(x - width / 2, stage_cases["squareOcc_drift_ratio"], width, label="squareOcc", color="#0072b2")
    axes[1, 1].bar(x + width / 2, stage_cases["IPR_drift_ratio"], width, label="IPR", color="#cc79a7")
    axes[1, 1].axhline(0.25, color="black", linestyle="--", linewidth=1.0)
    axes[1, 1].axhline(0.50, color="black", linestyle=":", linewidth=1.0)
    axes[1, 1].set_title("Gate drift / span")
    axes[1, 1].legend()
    for ax in axes.ravel():
        ax.set_xticks(x, labels, rotation=30, ha="right")
    plt.show()

elif mode == "tune":
    for case in summary["cases"]:
        rows = pd.DataFrame(case["rows"])
        rows["traj_len"] = rows["nfrog"] * rows["hmc_dt"]
        fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
        axes[0].scatter(rows["traj_len"], rows["acceptance_mean"], c=rows["hmc_mass"], cmap="viridis", s=80)
        axes[0].axhspan(0.70, 0.85, color="#d7ebff", alpha=0.8)
        axes[0].set_title(f"{case['name']} acceptance")
        axes[1].scatter(rows["traj_len"], rows["tau_int_doubleOcc_mean"], c=rows["hmc_mass"], cmap="viridis", s=80)
        axes[1].set_title("tau_int(doubleOcc)")
        axes[2].scatter(rows["traj_len"], rows["ess_per_sec_doubleOcc_mean"], c=rows["hmc_mass"], cmap="viridis", s=80)
        axes[2].set_title("ESS/sec")
        for ax in axes:
            ax.set_xlabel("Nfrog * dt")
        plt.show()

elif mode == "benchmark":
    labels = [case["name"] for case in summary["cases"]]
    accept = [case["acceptance_mean"] for case in summary["cases"]]
    ess_local = [case["perf"]["local"]["ess_per_sec_doubleOcc"] for case in summary["cases"]]
    ess_hmc = [case["perf"]["hmc"]["ess_per_sec_doubleOcc"] for case in summary["cases"]]
    tau_local = [case["perf"]["local"]["tau_int_doubleOcc"] for case in summary["cases"]]
    tau_hmc = [case["perf"]["hmc"]["tau_int_doubleOcc"] for case in summary["cases"]]
    x = np.arange(len(labels))
    fig, axes = plt.subplots(3, 1, figsize=(12, 12), constrained_layout=True)
    axes[0].bar(x, accept, color="#3b6fb6")
    axes[0].axhspan(0.70, 0.85, color="#d7ebff")
    axes[0].set_ylabel("Acceptance")
    axes[0].set_xticks(x, labels, rotation=30, ha="right")
    width = 0.38
    axes[1].bar(x - width/2, ess_local, width, label="Local", color="#999999")
    axes[1].bar(x + width/2, ess_hmc, width, label="HMC", color="#d55e00")
    axes[1].legend()
    axes[1].set_ylabel("ESS / sec")
    axes[1].set_xticks(x, labels, rotation=30, ha="right")
    axes[2].bar(x - width/2, tau_local, width, label="Local", color="#999999")
    axes[2].bar(x + width/2, tau_hmc, width, label="HMC", color="#009e73")
    axes[2].legend()
    axes[2].set_ylabel("tau_int(doubleOcc)")
    axes[2].set_xticks(x, labels, rotation=30, ha="right")
    plt.show()


In [ ]:
if mode == "stage":
    case_name = CASE_SELECT or stage_cases.iloc[0]["name"]
    case_df = stage_samples[stage_samples["name"] == case_name].copy()
    fig, axes = plt.subplots(len(TRACE_OBSERVABLES), 1, figsize=(12, 3 * len(TRACE_OBSERVABLES)), sharex=True, constrained_layout=True)
    if len(TRACE_OBSERVABLES) == 1:
        axes = [axes]
    for ax, observable in zip(axes, TRACE_OBSERVABLES):
        obs_df = case_df[case_df["observable"] == observable]
        for repeat, repeat_df in obs_df.groupby("repeat"):
            repeat_df = repeat_df.sort_values("sample_index")
            ax.plot(repeat_df["sample_index"], repeat_df["value"], label=f"rep {repeat}")
        ax.set_ylabel(observable)
        ax.grid(alpha=0.2)
        ax.legend(loc="best")
    axes[-1].set_xlabel("Sample index")
    axes[0].set_title(case_name)
    plt.show()
else:
    print("Trace panel is only defined for stage summaries.")
